# Sistema multi-agente Strands com execução paralela usando AgentCore Memory Branching 

## Introdução

Este notebook demonstra o **AgentCore Memory Branching** - uma capacidade poderosa que permite que múltiplos agentes especializados mantenham contextos de memória isolados enquanto compartilham um recurso de memória comum. Isso é essencial para sistemas multi-agente, especialmente aqueles que utilizam padrões de execução paralela como Strands Agent Graphs.

## Por Que o Memory Branching é Importante

Em sistemas multi-agente, diferentes agentes frequentemente precisam:
- **Manter contextos de conversa separados** - Cada agente foca em seu domínio sem interferência
- **Executar em paralelo** - Múltiplos agentes podem trabalhar simultaneamente sem conflitos de memória
- **Compartilhar uma sessão comum** - Todos os agentes contribuem para a mesma sessão do usuário mantendo seus contextos isolados
- **Acessar histórico relevante** - Agentes podem recuperar suas próprias interações passadas sem misturar contextos

O AgentCore Memory Branching resolve esses desafios permitindo múltiplas ramificações de conversa dentro de uma única sessão de memória, similar a branches do Git para código.

## Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo do tutorial    | Multi-Agente com Memory Branching                                                |
| Caso de uso         | Assistente de Planejamento de Viagens                                            |
| Framework agêntico  | Strands Agent Graph (suporta execução paralela)                                  |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes         | AgentCore Memory Branching, Strands Multi-Agent Graph, Execução Paralela         |
| Complexidade        | Intermediário                                                                    |


O que você aprenderá:

- Como criar e gerenciar branches de memória para diferentes agentes
- Implementar contextos de memória isolados em uma arquitetura multi-agente
- Construir grafos de agentes com Strands que suportam execução paralela
- Como o branching permite acesso concorrente seguro à memória
- Visualizar e inspecionar o histórico de conversas específico de cada branch

### Contexto do cenário

Vamos construir um **Sistema de Planejamento de Viagens** com três agentes, cada um com seu próprio branch de memória:
1. **Coordenador de Viagens** (branch main) - Orquestra o planejamento geral da viagem
2. **Assistente de Reserva de Voos** (branch flight_agent_memory) - Lida com consultas de viagens aéreas
3. **Assistente de Reserva de Hotéis** (branch hotel_agent_memory) - Gerencia solicitações de acomodação

O coordenador pode delegar para agentes especializados que executam em paralelo, com cada um mantendo seu próprio histórico de conversa através do memory branching.

## Arquitetura
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Pré-requisitos
- Python 3.10+
- Conta AWS com permissões apropriadas
- IAM role da AWS com permissões apropriadas para AgentCore Memory
- Acesso aos modelos do Amazon Bedrock

Vamos começar configurando nosso ambiente e criando nosso recurso de memória compartilhada com suporte a branching!

## Passo 1: Configuração do ambiente
Vamos começar importando todas as bibliotecas necessárias e definindo os clientes para que este notebook funcione.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

Defina a região e a role com as permissões apropriadas para os modelos do Amazon Bedrock e AgentCore

In [ ]:
import os
region = os.getenv('AWS_REGION', 'us-west-2')
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

## Passo 2: Criando Memória Compartilhada com Suporte a Branching

Vamos criar um único recurso de memória que suportará múltiplos branches - um para cada agente. Este recurso de memória compartilhada atua como a base, enquanto os branches fornecem contextos isolados para as conversas de cada agente.

Pense nisso como um repositório Git: um repositório (recurso de memória) com múltiplos branches (contextos dos agentes).

In [ ]:
from bedrock_agentcore.memory import MemoryClient

In [ ]:
client = MemoryClient(region_name=region)
memory_name = "TravelAgent_STM_%s" % datetime.now().strftime("%Y%m%d%H%M%S")
memory_id = None


In [ ]:
from botocore.exceptions import ClientError

try:
    print("Creating Memory...")
    memory_name = memory_name

    # Create the memory resource
    memory = client.create_memory_and_wait(
        name=memory_name,                       # Unique name for this memory store
        description="Travel Agent STM",         # Human-readable description
        strategies=[],                          # No special memory strategies for short-term memory
        event_expiry_days=7,                    # Memories expire after 7 days
        max_wait=300,                           # Maximum time to wait for memory creation (5 minutes)
        poll_interval=10                        # Check status every 10 seconds
    )

    # Extract and print the memory ID
    memory_id = memory['id']
    print(f"Memory created successfully with ID: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    print(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()

    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

### Entendendo o Memory Branching para Sistemas Multi-Agente

O recurso de memória que criamos suporta **branching** - uma funcionalidade crítica para arquiteturas multi-agente. Veja como funciona:

**Um Único Recurso de Memória, Múltiplos Branches:**
- Todos os agentes compartilham o mesmo `memory_id` e `session_id`
- Cada agente recebe seu próprio `branch_name` para contexto isolado

**Benefícios Principais para Sistemas Multi-Agente:**

1. **Isolamento de Contexto**: Cada agente mantém seu próprio histórico de conversa sem interferência
   - O agente de voos vê apenas conversas relacionadas a voos
   - O agente de hotéis vê apenas conversas relacionadas a hotéis
   - O coordenador vê o fluxo principal de orquestração

2. **Segurança na Execução Paralela**: Múltiplos agentes podem executar simultaneamente
   - Sem conflitos de memória quando agentes executam em paralelo
   - Cada branch é acessível independentemente
   - Crítico para Strands Agent Graphs que suportam execução concorrente

3. **Trilha de Auditoria Clara**: As interações de cada agente são rastreáveis
   - Inspecione o que cada agente discutiu
   - Depure problemas específicos de cada agente
   - Entenda o fluxo de conversas multi-agente

## Passo 3: Criar Memory Hook Provider com Suporte a Branch

A classe `ShortTermMemoryHook` implementa o gerenciamento de memória com reconhecimento de branches. Este é o componente-chave que habilita o memory branching em nosso sistema multi-agente.

**Funcionalidades Principais:**

1. **Inicialização de Branch**: Cria automaticamente branches para cada agente
   - Branch principal para o agente coordenador
   - Branches especializados (ex.: `flight_agent_memory`, `hotel_agent_memory`) para sub-agentes
   - Branches são criados a partir da linha do tempo principal da conversa

2. **Recuperação de Memória Específica por Branch**: Cada agente carrega apenas seu próprio contexto
   - `on_agent_initialized()` busca o histórico de conversa do branch do agente
   - Previne poluição de contexto entre agentes
   - Permite que agentes mantenham conversas focadas e específicas do domínio

3. **Armazenamento de Memória Específico por Branch**: Conversas são salvas no branch correto
   - `on_message_added()` armazena mensagens no branch designado do agente
   - Suporta escritas concorrentes da execução paralela de agentes
   - Sem condições de corrida ou conflitos de memória

Este hook provider é o que torna a execução paralela de agentes segura e eficiente com o AgentCore Memory.

In [ ]:
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory import MemorySessionManager
class ShortTermMemoryHook(HookProvider):
    def __init__(self, memory_id: str, region_name: str = "us-west-2", branch_name: str = "main"):
        """Initialize the hook with a MemorySessionManager.

        Args:
            memory_id: The AgentCore Memory ID
            region_name: AWS region for the memory service
            branch_name: Branch name for this agent's memory (default: "main")
        """
        self.memory_manager = MemorySessionManager(
            memory_id=memory_id,
            region_name=region_name
        )
        self.memory_id = memory_id
        self.branch_name = branch_name
        self._sessions = {}  # Cache session objects per actor/session combo
        self._branch_initialized = False  # Track if branch has been created

    def _get_or_create_session(self, actor_id: str, session_id: str):
        """Get or create a MemorySession for the given actor/session.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            MemorySession object
        """
        key = f"{actor_id}:{session_id}"
        if key not in self._sessions:
            self._sessions[key] = self.memory_manager.create_memory_session(
                actor_id=actor_id,
                session_id=session_id
            )
        return self._sessions[key]

    def _initialize_branch(self, actor_id: str, session_id: str):
        """Initialize a branch if it doesn't exist and this is not the main branch.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier
        """
        if self._branch_initialized or self.branch_name == "main":
            return

        try:
            memory_session = self._get_or_create_session(actor_id, session_id)

            # Check if branch already exists
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)

            if not branch_exists:
                # Get the last event from main branch to fork from
                main_events = memory_session.list_events(branch_name="main")
                if main_events:
                    last_event = main_events[-1]
                    # Create the branch with an initial message
                    memory_session.fork_conversation(
                        root_event_id=last_event.eventId,
                        branch_name=self.branch_name,
                        messages=[
                            ConversationalMessage(f"Starting {self.branch_name} branch", MessageRole.ASSISTANT)
                        ]
                    )
                    logger.info(f"✅ Created branch: {self.branch_name}")

            self._branch_initialized = True

        except Exception as e:
            logger.error(f"Failed to initialize branch {self.branch_name}: {e}", exc_info=True)

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load recent conversation history when agent starts"""
        try:
            # Get session info from agent state
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in agent state")
                return

            # Get the memory session
            memory_session = self._get_or_create_session(actor_id, session_id)

            # For non-main branches, initialize if there are events in main branch
            if self.branch_name != "main":
                try:
                    main_events = memory_session.list_events(branch_name="main")
                    if len(main_events) > 0:
                        self._initialize_branch(actor_id, session_id)
                except Exception as e:
                    # Main branch might not exist yet on first call
                    logger.info(f"Main branch not found yet, will initialize {self.branch_name} branch later: {e}")

            # Check if the branch exists before trying to get turns
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)
            
            recent_turns = []
            if branch_exists:
                # Only fetch turns if branch exists
                recent_turns = memory_session.get_last_k_turns(
                    k=5,
                    branch_name=self.branch_name
                )
            else:
                logger.info(f"Branch '{self.branch_name}' does not exist yet, skipping turn retrieval")

            if len(recent_turns) > 0:
                # Format conversation history for context
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message.get('role', 'unknown').lower()
                        text = message.get('content', {}).get('text', '')
                        if text:
                            context_messages.append(f"{role.title()}: {text}")

                if context_messages:
                    context = "\n".join(context_messages)
                    logger.info(f"Loaded context from branch '{self.branch_name}' ({len(context_messages)} messages)")

                    # Add context to agent's system prompt
                    event.agent.system_prompt += (
                        f"\n\nRecent conversation history (from {self.branch_name}):\n{context}\n\n"
                        "Continue the conversation naturally based on this context."
                    )

                    logger.info(f"✅ Loaded {len(recent_turns)} recent conversation turns from branch '{self.branch_name}'")
            else:
                logger.info(f"No previous conversation history found in branch '{self.branch_name}'")

        except Exception as e:
            logger.error(f"Failed to load conversation history: {e}", exc_info=True)

    def on_message_added(self, event: MessageAddedEvent):
        """Store conversation turns in memory on the appropriate branch"""
        try:
            # Get session info from agent state
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in agent state")
                return

            # Get the memory session
            memory_session = self._get_or_create_session(actor_id, session_id)

            # Get the last message
            messages = event.agent.messages
            if not messages:
                return

            last_message = messages[-1]
            role_str = last_message.get("role", "").upper()
            content_text = last_message.get("content", [{}])[0].get("text", "")

            if not content_text:
                logger.debug("Skipping empty message")
                return

            # Map role string to MessageRole enum
            role_mapping = {
                "USER": MessageRole.USER,
                "ASSISTANT": MessageRole.ASSISTANT,
                "TOOL": MessageRole.TOOL,
            }
            message_role = role_mapping.get(role_str, MessageRole.USER)

            # Store the message on the appropriate branch
            if self.branch_name == "main":
                # Main branch - just add turns normally
                memory_session.add_turns(
                    messages=[ConversationalMessage(content_text, message_role)]
                )
            else:
                # Non-main branch - need to append to existing branch
                # Initialize branch if it doesn't exist
                if not self._branch_initialized:
                    self._initialize_branch(actor_id, session_id)

                # Get the latest event from this branch
                branch_events = memory_session.list_events(branch_name=self.branch_name)
                if branch_events:
                    # Add to existing branch by specifying branch name (without rootEventId)
                    memory_session.add_turns(
                        messages=[ConversationalMessage(content_text, message_role)],
                        branch={"name": self.branch_name}
                    )
                else:
                    # This shouldn't happen if _initialize_branch worked, but handle it
                    logger.warning(f"Branch {self.branch_name} not found after initialization")
                    self._initialize_branch(actor_id, session_id)

            logger.debug(f"✅ Stored message in branch '{self.branch_name}': {role_str}")

        except Exception as e:
            logger.error(f"Failed to store message: {e}", exc_info=True)

    def create_branch(self, actor_id: str, session_id: str,
                      root_event_id: str, branch_name: str,
                      messages: list):
        """Create a new conversation branch.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier
            root_event_id: Event ID to branch from
            branch_name: Name for the new branch
            messages: List of ConversationalMessage objects to add to the branch
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.fork_conversation(
            root_event_id=root_event_id,
            branch_name=branch_name,
            messages=messages
        )

    def list_branches(self, actor_id: str, session_id: str):
        """List all branches for a session.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            List of branch information
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.list_branches()

    def get_session(self, actor_id: str, session_id: str):
        """Get the memory session object for direct access.

        Args:
            actor_id: The actor identifier
            session_id: The session identifier

        Returns:
            MemorySession object
        """
        return self._get_or_create_session(actor_id, session_id)

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register memory hooks with the registry.

        Args:
            registry: The HookRegistry to register callbacks with
        """
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## Passo 4: Criar Arquitetura Multi-Agente com Strands Agent Graph

Agora vamos construir nosso sistema multi-agente usando **Strands Agent Graph** - um framework que suporta execução paralela de agentes. Cada agente será configurado com seu próprio branch de memória, habilitando operação concorrente segura.

**Visão Geral da Arquitetura:**
- **Agente Coordenador** → Usa branch `main`
- **Agente de Voos** → Usa branch `flight_agent_memory`
- **Agente de Hotéis** → Usa branch `hotel_agent_memory`

Todos os agentes compartilham o mesmo `session_id` mas mantêm contextos de conversa isolados através do branching.

In [ ]:
# Import the necessary components
from strands import Agent, tool

In [ ]:
# Create unique actor IDs for each specialized agent but share the session ID
actor_id = f"travel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
namespace = f"travel/{actor_id}/preferences/"

### Criando Agentes Especializados com Memória Específica por Branch

Vamos definir os system prompts para nossos agentes especializados. Cada agente será configurado com seu próprio branch de memória, garantindo isolamento de conversa e habilitando execução paralela.

In [ ]:
# System prompt for the hotel booking specialist
HOTEL_BOOKING_PROMPT = f"""You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner."""

# System prompt for the flight booking specialist
FLIGHT_BOOKING_PROMPT = f"""You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner."""

In [ ]:
flight_memory_hooks = None
hotel_memory_hooks = None

### Implementando Agentes com Memory Branching

Cada agente especializado é configurado com:
- Um `branch_name` único para contexto de memória isolado
- O mesmo `memory_id` e `session_id` para gerenciamento de sessão compartilhada
- Um `ShortTermMemoryHook` que gerencia operações específicas por branch

**Detalhes Principais da Implementação:**
- `flight_booking_agent()` usa o branch: `flight_agent_memory`
- `hotel_booking_agent()` usa o branch: `hotel_agent_memory`

In [ ]:
def flight_booking_agent() -> Agent:

    global flight_memory_hooks
    try:
        if flight_memory_hooks is None:
            # Create hook with branch name "flight_agent_memory"
            flight_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="flight_agent_memory"
            )

        flight_agent = Agent(
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id}
        )

        
        return flight_agent
    except Exception as e:
        return f"Error in flight booking assistant: {str(e)}"

def hotel_booking_agent() -> Agent:

    global hotel_memory_hooks
    try:
        if hotel_memory_hooks is None:
            # Create hook with branch name "hotel_agent_memory"
            hotel_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="hotel_agent_memory"
            )

        hotel_booking_agent = Agent(
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id}
        )

        return hotel_booking_agent
    except Exception as e:
        return f"Error in hotel booking assistant: {str(e)}"

### Criando o Agente Coordenador

O agente coordenador usa o branch `main` (padrão) e orquestra os agentes especializados. Ele pode delegar tarefas para os agentes de voos e hotéis, que podem executar em paralelo ao usar o Strands Agent Graph.

In [ ]:
# System prompt for the coordinator agent
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_agent
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_agent
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
def travel_booking_agent() -> Agent:

    agent_memory_hooks = ShortTermMemoryHook(
                    memory_id=memory_id,
                    region_name=region,
                )
    travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    hooks=[agent_memory_hooks],
    model=MODEL_ID,
    state={
        "actor_id": actor_id,
        "session_id": session_id
        }
    )

    return travel_agent

### Construindo o Agent Graph com Suporte a Execução Paralela

Agora vamos montar nossos agentes em um **Strands Agent Graph**. Esta estrutura de grafo permite:

**Execução Paralela:**
- Quando o coordenador precisa de informações de voos e hotéis, ambos os agentes podem executar simultaneamente
- O memory branching previne conflitos durante a execução concorrente
- Cada agente lê/escreve em seu próprio branch independentemente

**Mapeamento de Branches de Memória:**
```
Session: travel-session-xxx
├── main branch              → Travel Coordinator
├── flight_agent_memory      → Flight Booking Agent
└── hotel_agent_memory       → Hotel Booking Agent
```

**Por Que Isso é Importante:**
- Sem branching, agentes paralelos sobrescreveriam a memória um do outro
- Com branching, cada agente mantém seu próprio thread de conversa
- O coordenador pode delegar com segurança para múltiplos agentes ao mesmo tempo

In [ ]:
import logging
from strands import Agent
from strands.multiagent import GraphBuilder

# Enable debug logs and print them to stderr
logging.getLogger("strands.multiagent").setLevel(logging.DEBUG)
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

# Build the Strands Agent Graph
# This graph structure enables parallel execution of specialized agents
# Memory branching ensures safe concurrent access without conflicts
builder = GraphBuilder()

# Add nodes - each agent with its own memory branch
builder.add_node(travel_booking_agent(), "travel_agent")           # Uses 'main' branch
builder.add_node(flight_booking_agent(), "flight_booking_agent")   # Uses 'flight_agent_memory' branch
builder.add_node(hotel_booking_agent(), "hotel_booking_agent")     # Uses 'hotel_agent_memory' branch

# Add edges - define which agents the coordinator can delegate to
# The graph can execute flight and hotel agents in parallel when both are needed
builder.add_edge("travel_agent", "flight_booking_agent")
builder.add_edge("travel_agent", "hotel_booking_agent")

# Set entry point - the coordinator agent receives user input first
builder.set_entry_point("travel_agent")

# Configure execution limits for safety
builder.set_execution_timeout(600)   # 10 minute timeout

# Build the graph - ready for parallel execution with isolated memory contexts
graph = builder.build()


### O Sistema Multi-Agente com Memory Branching está Pronto!

Nosso grafo de agentes agora está configurado com:
- ✅ Três agentes com branches de memória isolados
- ✅ Capacidade de execução paralela através do Strands Agent Graph
- ✅ Acesso concorrente seguro à memória via AgentCore Memory Branching
- ✅ Criação e gerenciamento automático de branches

## Testando o Sistema Multi-Agente

Vamos testar com um cenário de planejamento de viagem que acionará múltiplos agentes:

In [ ]:
response = graph("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

## Inspecionando Branches de Memória

Uma das principais vantagens do AgentCore Memory Branching é a capacidade de inspecionar o histórico de conversas de cada agente independentemente. Isso é crucial para:

**Depuração de Sistemas Multi-Agente:**
- Veja exatamente o que cada agente discutiu
- Identifique qual agente lidou com qual parte da conversa
- Rastreie o fluxo de informações através do sistema

**Entendendo a Execução Paralela:**
- Verifique que os agentes mantiveram contextos separados
- Confirme que não houve conflitos de memória durante a execução concorrente
- Audite a linha do tempo das interações dos agentes

Vamos explorar os branches que foram criados durante nossa conversa:

In [ ]:
print("\n=== Viewing Memory Branches ===")

if flight_memory_hooks or hotel_memory_hooks:
    # Get any memory session to list branches (they all point to the same session)
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # List all branches in the session
        branches = memory_session.list_branches()
        print(f"\n📊 Session has {len(branches)} branches total:")
        for branch in branches:
            print(f"  - Branch: {branch.name}")
            print(f"    └─ Events: {len(memory_session.list_events(branch_name=branch.name))}")
            print(f"    └─ Created: {branch.created}")

        print("\n💡 Each branch represents a different agent's memory:")
        print("  • 'main' = Travel coordinator conversations")
        print("  • 'flight_agent_memory' = Flight assistant conversations")
        print("  • 'hotel_agent_memory' = Hotel assistant conversations")

### Acessando o Histórico de Conversas Específico por Branch

Agora vamos nos aprofundar e examinar as conversas reais armazenadas em cada branch. Isso demonstra como o memory branching fornece isolamento completo entre agentes mantendo uma sessão compartilhada.

In [ ]:
print("\n=== Accessing Branch-Specific Events ===")

if flight_memory_hooks or hotel_memory_hooks:
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # Get events from the main branch (coordinator)
        main_events = memory_session.list_events(branch_name="main")
        print(f"\n🌳 Main Branch - Coordinator ({len(main_events)} events):")
        if main_events:
            for event in main_events[-3:]:  # Show last 3 events
                for payload in event.payload:
                    if 'conversational' in payload:
                        role = payload['conversational']['role']
                        text = payload['conversational']['content']['text']
                        print(f"  {role}: {text[:100]}...")
        else:
            print("  No events found in main branch")

        # Get events from the flight agent branch
        try:
            flight_branch_events = memory_session.list_events(branch_name="flight_agent_memory")
            print(f"\n✈️  Flight Agent Branch ({len(flight_branch_events)} events):")
            if flight_branch_events:
                print("All flight-related conversations are stored here:")
                for event in flight_branch_events[-3:]:  # Show last 3 events
                    for payload in event.payload:
                        if 'conversational' in payload:
                            role = payload['conversational']['role']
                            text = payload['conversational']['content']['text']
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  No events found - flight assistant wasn't called yet")
        except Exception as e:
            print(f"  Flight branch not created yet: {e}")

        # Get events from the hotel agent branch
        try:
            hotel_branch_events = memory_session.list_events(branch_name="hotel_agent_memory")
            print(f"\n🏨 Hotel Agent Branch ({len(hotel_branch_events)} events):")
            if hotel_branch_events:
                print("All hotel-related conversations are stored here:")
                for event in hotel_branch_events[-3:]:  # Show last 3 events
                    for payload in event.payload:
                        if 'conversational' in payload:
                            role = payload['conversational']['role']
                            text = payload['conversational']['content']['text']
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  No events found - hotel assistant wasn't called yet")
        except Exception as e:
            print(f"  Hotel branch not created yet: {e}")

## Resumo

Neste notebook, demonstramos o **AgentCore Memory Branching** - uma capacidade crítica para construir sistemas multi-agente robustos com execução paralela:

### Principais Conclusões:

1. **Memory Branching Habilita Execução Paralela**
   - Múltiplos agentes podem executar simultaneamente sem conflitos de memória
   - Cada agente mantém seu próprio contexto de conversa através de branches
   - Essencial para Strands Agent Graphs e outros frameworks de agentes paralelos

2. **Isolamento de Contexto Melhora o Desempenho dos Agentes**
   - Agentes especializados focam em seu domínio sem interferência
   - Sem poluição de contexto entre agentes
   - Conversas mais limpas e relevantes

3. **Sessão Compartilhada com Contextos Isolados**
   - Único recurso de memória e session ID
   - Múltiplos branches para diferentes agentes
   - Utilização eficiente de recursos


### Padrão de Arquitetura:

```
Memory Resource (memory_id)
  └── Session (session_id)
      ├── main branch → Coordinator Agent
      ├── flight_agent_memory → Flight Agent (pode executar em paralelo)
      └── hotel_agent_memory → Hotel Agent (pode executar em paralelo)
```

O AgentCore Memory Branching torna seguro e eficiente construir sistemas multi-agente sofisticados que podem escalar conforme as necessidades da sua aplicação.

## Limpeza
Vamos deletar a memória para limpar os recursos utilizados neste notebook.

In [ ]:
#client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)